# INSTALL

In [ ]:
!pip install typhoon-ocr pdf2image Pillow

In [ ]:
!apt-get install -y poppler-utils

In [ ]:
!pip install -U google-generativeai

In [ ]:
!pip install python-dotenv

In [5]:
# !pip uninstall -y transformers tokenizers huggingface_hub accelerate
!pip install transformers accelerate tiktoken einops

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached tiktoken-0.12.0-cp314-cp314-win_amd64.whl.metadata (6.9 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ---------------------------------------- 0.0/921.1 kB ? eta -:--:--
   ---------------------------------------- 921.1/921.1 kB 10.1 MB/s  0:00:00
Using cached einops-0.8.2-py3-none-any.whl (65 kB)

   -------------------- ------------------- 2/4 [accelerate]
   -------------------- ------------------- 2/4 [accelerate]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ ---

In [2]:
!pip uninstall -y google-generativeai
!pip install google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
  Using cached google_genai-1.70.0-py3-none-any.whl.metadata (52 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
Using cached google_genai-1.70.0-py3-none-any.whl (760 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)

   ------------- -------------------------- 1/3 [tenacity]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   ---------------------------------------- 3/3 [google-genai]



In [6]:
import transformers
print(f"Transformers version: {transformers.__version__}")

c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.5.0


# IMPORT

In [ ]:
import os
import json
import re
import requests
# import dashscope
import shutil
from pdf2image import convert_from_path
from typhoon_ocr import ocr_document
from google import genai
from google.genai import types
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Pipeline

In [ ]:
class Extractor:
    # รายชื่อพรรคการเมืองที่ถูกต้องสำหรับใช้ตรวจสอบ (Validation)
    VALID_PARTIES = """
    ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เครือข่ายชาวนาแห่งประเทศไทย, เพื่อไทย, 
    ชาติพัฒนา, ชาติไทยพัฒนา, อนาคตไทย, ภูมิใจไทย, สังคมประชาธิปไตยไทย, รักชาติ, 
    ประชาธิปไตยใหม่, พลังบูรพา, ครูไทยเพื่อประชาชน, พลังท้องถิ่นไท, ประชาชน, 
    ไทยก้าวใหม่, เสรีรวมไทย, รักษ์ธรรม, พลังประชาธิปไตย, พลังสุราษฎร์, พลังไทยรักชาติ, 
    เพื่อชีวิตใหม่, ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม, 
    รวมพลัง, ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, พร้อมพัฒนา, ประชาชาติ, แผ่นดินธรรม, 
    คลองไทย, พลังประชารัฐ, เศรษฐกิจใหม่, พลังสังคม, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, 
    กรีน, วิชชั่นใหม่, พลวัต, กล้าธรรม, ไทยรวมไทย, กล้า, ฟิวชัน, พลังสังคมใหม่, 
    ไทยสร้างไทย, รวมไทยสร้างชาติ, มิติใหม่, ไทยสมาร์ท, ไทยภักดี, ไทยพิทักษ์ธรรม, 
    ไทยชนะ, ไทรวมพลัง, ราษฎร์วิถี, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, 
    ไทยก้าวหน้า, ตะวันใหม่, พร้อม, รวมใจไทย, สัมมาธิปไตย, รักภูเก็ต, ประชาอาสาชาติ, 
    ไทยทรัพย์ทวี, รวมพลังประชาชน, อนาคตไกล, ยางพาราไทย, เพื่อบ้านเมือง
    """

    def __init__(self, typhoon_key: str, gemini_key: str):
        """Initializes the OCR engine and the Gemini LLM."""
        # Setup Typhoon OCR
        os.environ["TYPHOON_OCR_API_KEY"] = typhoon_key
        
        # Setup Gemini
        if not gemini_key:
            raise ValueError("GEMINI_KEY is missing. Please check your .env file.")
        self.gemini_client = genai.Client(api_key=gemini_key)

    def extract_pdf(self, pdf_path: str) -> str:
        """Process PDF: Multi-page OCR -> Merged Text -> LLM Parsing."""
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"The file {pdf_path} was not found.")
        
        print(f"Starting Extraction: {pdf_path}")
        
        temp_dir = "temp_pages"
        if not os.path.exists(temp_dir):
            os.makedirs(temp_dir)
            
        full_markdown_text = ""
        
        try:
            # แปลง PDF เป็นภาพทีละหน้าเพื่อส่งให้ OCR
            pages = convert_from_path(pdf_path)
            
            for i, page in enumerate(pages):
                page_path = os.path.join(temp_dir, f"page_{i+1}.jpg")
                page.save(page_path, "JPEG")
                
                print(f"--- OCRing Page {i+1}/{len(pages)} ---")
                page_markdown = ocr_document(page_path)
                full_markdown_text += f"\n--- PAGE {i+1} ---\n" + page_markdown
            
            print("Parsing with Gemini...")
            return self._parse_markdown_byGemini(full_markdown_text)
            
        finally:
            # ลบไฟล์ภาพชั่วคราว
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)

    def _get_shared_prompt(self, full_text: str) -> str:
        """Prompt logic for extracting data."""
        return f"""
        Extract the election results from the following Thai OCR text into a structured JSON format.
        
        Requirements:
        1. Fix any OCR typos in province, district, or party names.
        2. Convert all numbers (including Thai digits) to standard integers.
        3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
           and a 'results' list (party number, name, and votes).
        4. **PARTY NAME VALIDATION**: Compare the party name from OCR with the following valid list. 
           If the OCR name is misspelled, change it to the correct name from this list:
           {self.VALID_PARTIES}
        5. Check the score: look for numbers and Thai text in parentheses (). 
           If the main digit is unreadable, convert the Thai text description into an integer.

        OCR Text:
        {full_text}
        """

    def _parse_markdown_byGemini(self, full_text: str) -> str:
        prompt = self._get_shared_prompt(full_text)
        
        max_retries = 3
        for attempt in range(max_retries):
            try:
                # พยายามเรียก API
                response = self.gemini_client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json"
                    )
                )
                return response.text
                
            except Exception as e:
                # ถ้าเป็น Error 503 เซิร์ฟเวอร์ไม่ว่าง
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    if attempt < max_retries - 1:
                        wait_time = 3 * (attempt + 1)
                        print(f"⚠️ Google Server is busy. Retrying in {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                        time.sleep(wait_time)
                    else:
                        raise Exception("❌ Google API is down right now. Please try again later.")
                else:
                    raise e

In [8]:
if __name__ == "__main__":

    load_dotenv()
    
    TYPHOON_KEY = os.getenv("TYPHOON_KEY")
    GEMINI_KEY = os.getenv("GEMINI_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    extractor = Extractor(TYPHOON_KEY, GEMINI_KEY)
    
    try:
        json_output = extractor.extract_pdf("doc2.pdf")
        print("--- Extraction Result ---")
        print(json_output)
        
        # Save locally
        with open("output_gemini.json", "w", encoding="utf-8") as f:
            f.write(json_output)
            
    except Exception as e:
        print(f"An error occurred: {e}")

Starting Extraction: doc2.pdf
--- OCRing Page 1/4 ---
--- OCRing Page 2/4 ---
--- OCRing Page 3/4 ---
--- OCRing Page 4/4 ---
Parsing with Gemini...
An error occurred: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
